# Value at Risk (VaR) — A Practical Guide

**Author:** Ravi  
**Topics:** Historical VaR · Parametric VaR · Monte Carlo VaR · Backtesting

---

## What is VaR?

**Value at Risk (VaR)** answers the question:

> *"What is the maximum loss I can expect over a given time horizon, at a given confidence level?"*

For example, a **1-day 99% VaR of \$1M** means:  
*"There is a 1% chance of losing more than \$1M in a single day."*

We will implement and compare three methods:

| Method | Assumption | Strength |
|--------|-----------|----------|
| **Historical** | None — uses actual past returns | Captures fat tails, no distributional assumption |
| **Parametric** | Returns are normally distributed | Fast, analytical, widely used |
| **Monte Carlo** | Specified distribution (e.g. normal) | Flexible, handles complex portfolios |

We finish with **backtesting** — checking whether our VaR model actually holds up.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

# Parameters
CONFIDENCE = 0.99          # 99% VaR
HORIZON    = 1             # 1-day holding period
PORTFOLIO_VALUE = 1_000_000  # $1M portfolio
LOOKBACK   = 252           # 1 trading year of history for estimation window
N_SIMULATIONS = 10_000     # Monte Carlo paths

print(f"VaR parameters: {int(CONFIDENCE*100)}% confidence | {HORIZON}-day horizon | ${PORTFOLIO_VALUE:,.0f} portfolio")

## 1. Data

We build a simple **equal-weight portfolio** of four large-cap stocks:  
`AAPL, MSFT, JPM, XOM`

Using ~5 years of daily closing prices from Yahoo Finance.

In [ ]:
TICKERS  = ['AAPL', 'MSFT', 'JPM', 'XOM']
WEIGHTS  = np.array([0.25, 0.25, 0.25, 0.25])

raw = yf.download(TICKERS, start='2019-01-01', end='2024-12-31', auto_adjust=True)['Close']
prices = raw.dropna()

# Daily log returns
returns = np.log(prices / prices.shift(1)).dropna()

# Portfolio daily return (weighted sum of log returns — valid for short horizons)
port_returns = returns @ WEIGHTS

print(f"Date range : {returns.index[0].date()} → {returns.index[-1].date()}")
print(f"Trading days: {len(returns)}")
print(f"\nPortfolio daily return stats:")
print(port_returns.describe().round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Cumulative portfolio return
cum_ret = (1 + port_returns).cumprod()
axes[0].plot(cum_ret.index, cum_ret.values, color='steelblue', lw=1.5)
axes[0].set_title('Portfolio Cumulative Return')
axes[0].set_ylabel('Growth of $1')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.2f}'))

# Return distribution
axes[1].hist(port_returns, bins=80, color='steelblue', alpha=0.7, edgecolor='white', density=True)
x = np.linspace(port_returns.min(), port_returns.max(), 300)
axes[1].plot(x, stats.norm.pdf(x, port_returns.mean(), port_returns.std()),
             'r--', lw=2, label='Normal fit')
axes[1].set_title('Daily Return Distribution')
axes[1].set_xlabel('Log Return')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

# Quick normality check
stat, p = stats.shapiro(port_returns.sample(500, random_state=42))
print(f"Shapiro-Wilk test  p-value: {p:.4f}  → returns {'are NOT' if p < 0.05 else 'appear'} normally distributed")

kurt = port_returns.kurtosis()
print(f"Excess kurtosis: {kurt:.2f}  (0 = normal; positive = fat tails)")

## 2. Historical VaR

**Idea:** Sort the last *N* days of actual portfolio returns. VaR is the return at the (1 − confidence) percentile.

$$\text{VaR}_{\alpha} = -\text{Percentile}(r_1, r_2, \ldots, r_N, \; 1-\alpha)$$

**No distributional assumptions.** Captures fat tails exactly as they appeared historically.

In [ ]:
def historical_var(returns_series, confidence=0.99, portfolio_value=1_000_000):
    """Historical (non-parametric) VaR."""
    cutoff = np.percentile(returns_series, (1 - confidence) * 100)
    var_pct = -cutoff                        # expressed as a positive loss
    var_dollar = var_pct * portfolio_value
    return var_pct, var_dollar, cutoff

# Use the full history for a single estimate
hist_var_pct, hist_var_dollar, cutoff_return = historical_var(
    port_returns, CONFIDENCE, PORTFOLIO_VALUE
)

print(f"Historical VaR ({int(CONFIDENCE*100)}%, 1-day)")
print(f"  Return threshold : {cutoff_return:.4f} ({cutoff_return*100:.2f}%)")
print(f"  VaR (% of portfolio): {hist_var_pct*100:.2f}%")
print(f"  VaR (dollar)        : ${hist_var_dollar:,.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(port_returns, bins=80, color='steelblue', alpha=0.6, edgecolor='white', density=True, label='Daily returns')

# Shade the tail
tail_x = np.linspace(port_returns.min(), cutoff_return, 200)
tail_y, _ = np.histogram(port_returns[port_returns <= cutoff_return], bins=40, density=True)
ax.axvline(cutoff_return, color='crimson', lw=2, linestyle='--', label=f'VaR threshold ({cutoff_return*100:.2f}%)')

# Annotate
ax.fill_betweenx(
    [0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 50],
    port_returns.min(), cutoff_return,
    alpha=0.25, color='crimson', label=f'1% worst losses'
)
ax.set_title(f'Historical VaR — {int(CONFIDENCE*100)}% Confidence Level', fontsize=13)
ax.set_xlabel('Daily Log Return')
ax.set_ylabel('Density')
ax.legend()
ax.annotate(
    f'VaR = ${hist_var_dollar:,.0f}\n({hist_var_pct*100:.2f}%)',
    xy=(cutoff_return, 3), xytext=(cutoff_return - 0.03, 15),
    arrowprops=dict(arrowstyle='->', color='crimson'),
    color='crimson', fontsize=10
)
plt.tight_layout()
plt.show()

## 3. Parametric (Variance-Covariance) VaR

**Assumption:** Portfolio returns are normally distributed with mean $\mu$ and standard deviation $\sigma$.

$$\text{VaR}_{\alpha} = -(\mu + z_{\alpha} \cdot \sigma)$$

where $z_{\alpha}$ is the $\alpha$-th quantile of the standard normal (e.g. $z_{0.01} = -2.326$ for 99% VaR).

For a multi-asset portfolio:
$$\sigma_p = \sqrt{\mathbf{w}^\top \Sigma \mathbf{w}}$$

where $\Sigma$ is the covariance matrix of asset returns.

In [ ]:
def parametric_var(returns_df, weights, confidence=0.99, portfolio_value=1_000_000):
    """Variance-Covariance (parametric, normal) VaR."""
    mu    = (returns_df @ weights).mean()
    cov   = returns_df.cov().values
    sigma = np.sqrt(weights @ cov @ weights)
    z     = stats.norm.ppf(1 - confidence)   # negative number
    var_pct    = -(mu + z * sigma)
    var_dollar = var_pct * portfolio_value
    return var_pct, var_dollar, mu, sigma

para_var_pct, para_var_dollar, mu_p, sigma_p = parametric_var(
    returns, WEIGHTS, CONFIDENCE, PORTFOLIO_VALUE
)

z_99 = stats.norm.ppf(1 - CONFIDENCE)

print(f"Parametric VaR ({int(CONFIDENCE*100)}%, 1-day)")
print(f"  Portfolio daily mean  (μ): {mu_p:.6f} ({mu_p*100:.4f}%)")
print(f"  Portfolio daily vol   (σ): {sigma_p:.6f} ({sigma_p*100:.4f}%)")
print(f"  z-score at {int(CONFIDENCE*100)}%          : {z_99:.4f}")
print(f"  VaR (% of portfolio)      : {para_var_pct*100:.2f}%")
print(f"  VaR (dollar)              : ${para_var_dollar:,.0f}")

In [ ]:
# Show the covariance matrix and correlation matrix
cov_matrix  = returns.cov() * 252          # annualised
corr_matrix = returns.corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, matrix, title, fmt in zip(
    axes,
    [cov_matrix * 100, corr_matrix],
    ['Annualised Covariance (×100)', 'Correlation Matrix'],
    ['.3f', '.2f']
):
    im = ax.imshow(matrix, cmap='RdYlGn', aspect='auto',
                   vmin=-matrix.abs().max().max(), vmax=matrix.abs().max().max())
    ax.set_xticks(range(len(TICKERS))); ax.set_xticklabels(TICKERS)
    ax.set_yticks(range(len(TICKERS))); ax.set_yticklabels(TICKERS)
    ax.set_title(title)
    for i in range(len(TICKERS)):
        for j in range(len(TICKERS)):
            ax.text(j, i, format(matrix.iloc[i, j], fmt),
                    ha='center', va='center', fontsize=9)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

## 4. Monte Carlo VaR

**Idea:** Simulate thousands of possible portfolio return paths, then read off the empirical tail.

We model the joint return distribution using a **multivariate normal** (Cholesky decomposition to preserve correlations):

$$\mathbf{r}_{sim} = \boldsymbol{\mu} + L \cdot \mathbf{z}, \quad \mathbf{z} \sim \mathcal{N}(0, I)$$

where $L$ is the Cholesky factor of $\Sigma$ ($\Sigma = LL^\top$).

The portfolio return for each simulation is $r_p = \mathbf{w}^\top \mathbf{r}_{sim}$.

In [ ]:
def monte_carlo_var(returns_df, weights, n_sims=10_000, confidence=0.99,
                    portfolio_value=1_000_000, seed=42):
    """Monte Carlo VaR using multivariate normal simulation."""
    rng   = np.random.default_rng(seed)
    mu    = returns_df.mean().values
    cov   = returns_df.cov().values
    L     = np.linalg.cholesky(cov)         # Cholesky decomposition

    # Simulate n_sims joint daily returns
    z            = rng.standard_normal((n_sims, len(weights)))
    sim_returns  = mu + z @ L.T             # shape: (n_sims, n_assets)
    port_sim_ret = sim_returns @ weights    # shape: (n_sims,)

    cutoff     = np.percentile(port_sim_ret, (1 - confidence) * 100)
    var_pct    = -cutoff
    var_dollar = var_pct * portfolio_value
    return var_pct, var_dollar, port_sim_ret

mc_var_pct, mc_var_dollar, mc_sim_returns = monte_carlo_var(
    returns, WEIGHTS, N_SIMULATIONS, CONFIDENCE, PORTFOLIO_VALUE
)

print(f"Monte Carlo VaR ({int(CONFIDENCE*100)}%, 1-day, {N_SIMULATIONS:,} simulations)")
print(f"  VaR (% of portfolio): {mc_var_pct*100:.2f}%")
print(f"  VaR (dollar)        : ${mc_var_dollar:,.0f}")

In [ ]:
mc_cutoff = np.percentile(mc_sim_returns, (1 - CONFIDENCE) * 100)

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(mc_sim_returns, bins=100, color='darkorange', alpha=0.6, edgecolor='white',
        density=True, label=f'{N_SIMULATIONS:,} MC simulations')
ax.axvline(mc_cutoff, color='crimson', lw=2, linestyle='--',
           label=f'MC VaR threshold ({mc_cutoff*100:.2f}%)')

# Overlay normal fit
x = np.linspace(mc_sim_returns.min(), mc_sim_returns.max(), 400)
ax.plot(x, stats.norm.pdf(x, mc_sim_returns.mean(), mc_sim_returns.std()),
        'k--', lw=1.5, label='Normal fit')

ax.set_title(f'Monte Carlo Simulated Portfolio Returns  ({N_SIMULATIONS:,} paths)', fontsize=13)
ax.set_xlabel('1-Day Log Return')
ax.set_ylabel('Density')
ax.legend()
ax.annotate(
    f'MC VaR = ${mc_var_dollar:,.0f}\n({mc_var_pct*100:.2f}%)',
    xy=(mc_cutoff, 3), xytext=(mc_cutoff - 0.025, 18),
    arrowprops=dict(arrowstyle='->', color='crimson'),
    color='crimson', fontsize=10
)
plt.tight_layout()
plt.show()

## 5. Method Comparison

In [ ]:
results = pd.DataFrame({
    'Method'       : ['Historical', 'Parametric', 'Monte Carlo'],
    'VaR (%)'      : [hist_var_pct*100, para_var_pct*100, mc_var_pct*100],
    'VaR ($)'      : [hist_var_dollar, para_var_dollar, mc_var_dollar],
    'Assumption'   : ['None (empirical)', 'Multivariate Normal', 'Multivariate Normal']
}).set_index('Method')

results['VaR (%)'] = results['VaR (%)'].map('{:.2f}%'.format)
results['VaR ($)'] = results['VaR ($)'].map('${:,.0f}'.format)

print(f"Portfolio: {TICKERS}  |  Equal-weight  |  ${PORTFOLIO_VALUE:,.0f}")
print(f"Confidence: {int(CONFIDENCE*100)}%  |  Horizon: {HORIZON}-day\n")
print(results.to_string())

In [ ]:
methods = ['Historical', 'Parametric', 'Monte Carlo']
dollars = [hist_var_dollar, para_var_dollar, mc_var_dollar]
colors  = ['steelblue', 'seagreen', 'darkorange']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(methods, dollars, color=colors, edgecolor='white', width=0.5)

for bar, val in zip(bars, dollars):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'${val:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title(f'{int(CONFIDENCE*100)}% 1-Day VaR Comparison  (${PORTFOLIO_VALUE:,.0f} portfolio)', fontsize=13)
ax.set_ylabel('VaR ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_ylim(0, max(dollars) * 1.2)
plt.tight_layout()
plt.show()

## 6. Backtesting VaR

Backtesting checks: **how often did actual losses exceed our VaR?**

We use a **rolling window** approach:
- Estimate VaR each day using the previous `LOOKBACK` days
- Check whether the *next* day's actual return exceeds that VaR
- A **VaR breach** (exception) occurs when: $r_{t+1} < -\text{VaR}_t$

For a well-calibrated 99% VaR, we expect breaches ~**1% of the time**.

We test this formally with **Kupiec's Proportion of Failures (POF) test**:

$$LR_{POF} = -2\ln\left[\frac{(1-\alpha)^{T-x}\alpha^x}{(1-\hat{p})^{T-x}\hat{p}^x}\right] \sim \chi^2(1)$$

where $x$ = number of exceptions, $T$ = total observations, $\hat{p} = x/T$.

In [ ]:
def rolling_historical_var(port_returns, lookback=252, confidence=0.99):
    """Compute rolling 1-day Historical VaR (returns as fraction)."""
    var_series = (
        port_returns
        .rolling(window=lookback)
        .quantile(1 - confidence)
        .mul(-1)
    )
    return var_series

def kupiec_test(exceptions, n_obs, confidence=0.99):
    """Kupiec POF likelihood ratio test. Returns (LR statistic, p-value)."""
    alpha = 1 - confidence
    x, T  = exceptions, n_obs
    p_hat = x / T
    if p_hat == 0 or p_hat == 1:
        return np.nan, np.nan
    lr = -2 * (x * np.log(alpha / p_hat) + (T - x) * np.log((1 - alpha) / (1 - p_hat)))
    p_val = 1 - stats.chi2.cdf(lr, df=1)
    return lr, p_val

# Rolling VaR series
rolling_var = rolling_historical_var(port_returns, LOOKBACK, CONFIDENCE)

# Align: compare today's VaR forecast to tomorrow's actual return
var_forecast = rolling_var.shift(1)          # yesterday's VaR estimate
actual_loss  = -port_returns                 # actual loss (positive = lost money)

# Drop NaN rows (first LOOKBACK days)
valid = var_forecast.dropna().index
var_f = var_forecast.loc[valid]
act_l = actual_loss.loc[valid]

# Exceptions = days where actual loss > VaR forecast
exceptions  = (act_l > var_f)
n_exceptions = exceptions.sum()
n_total      = len(valid)
breach_rate  = n_exceptions / n_total

lr_stat, p_val = kupiec_test(n_exceptions, n_total, CONFIDENCE)

print("=" * 50)
print("Backtest Results (Historical VaR, Rolling 252-day)")
print("=" * 50)
print(f"  Observation window  : {valid[0].date()} → {valid[-1].date()}")
print(f"  Total trading days  : {n_total}")
print(f"  VaR exceptions      : {n_exceptions}")
print(f"  Observed breach rate: {breach_rate*100:.2f}%  (expected ≤ {(1-CONFIDENCE)*100:.0f}%)")
print(f"\n  Kupiec POF Test")
print(f"    LR statistic  : {lr_stat:.4f}")
print(f"    p-value       : {p_val:.4f}")
print(f"    Model {'PASSES ✓' if p_val > 0.05 else 'FAILS ✗'} at 5% significance  (p {'>' if p_val > 0.05 else '<'} 0.05)")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# --- Top panel: portfolio P&L vs VaR band ---
axes[0].plot(act_l.index, act_l.values * 100, color='steelblue', lw=0.8,
             alpha=0.7, label='Daily loss (%)')
axes[0].plot(var_f.index, var_f.values * 100, color='crimson', lw=1.5,
             linestyle='--', label='Rolling 99% VaR')
axes[0].scatter(act_l.index[exceptions], act_l[exceptions].values * 100,
                color='red', s=30, zorder=5, label=f'VaR breach ({n_exceptions})')
axes[0].axhline(0, color='black', lw=0.5)
axes[0].set_title('Portfolio Daily Loss vs Rolling Historical VaR', fontsize=12)
axes[0].set_ylabel('Loss (%)')
axes[0].legend(loc='upper left')

# --- Bottom panel: cumulative exceptions ---
cum_exc  = exceptions.cumsum()
expected = pd.Series(
    np.arange(1, len(valid)+1) * (1 - CONFIDENCE),
    index=valid
)
axes[1].plot(cum_exc.index, cum_exc.values, color='crimson', lw=1.5,
             label='Actual cumulative exceptions')
axes[1].plot(expected.index, expected.values, color='grey', lw=1.5,
             linestyle='--', label='Expected (1% of days)')
axes[1].fill_between(cum_exc.index, expected.values * 0.5, expected.values * 1.5,
                     alpha=0.15, color='grey', label='±50% tolerance band')
axes[1].set_title('Cumulative VaR Exceptions vs Expected', fontsize=12)
axes[1].set_ylabel('Cumulative exceptions')
axes[1].set_xlabel('Date')
axes[1].legend(loc='upper left')

plt.tight_layout()
plt.show()

## 7. Key Takeaways

| | Historical | Parametric | Monte Carlo |
|--|--|--|--|
| **Distributional assumption** | None | Normal | Normal (or any) |
| **Fat tails captured?** | Yes — directly | No — underestimates | Depends on distribution chosen |
| **Computation** | Fast | Analytical (fastest) | Slow (scales with N) |
| **Best used when** | Data is rich, tails matter | Quick daily calculation | Complex products, non-linear payoffs |

### Limitations of VaR

1. **VaR does not tell you how bad the loss will be** *beyond* the threshold — that is what **Expected Shortfall (CVaR)** addresses.
2. Historical VaR is backward-looking — a quiet market period will produce artificially low estimates.
3. Normal VaR systematically underestimates tail risk (as confirmed by the fat tails observed above).
4. Basel III now requires banks to use **Expected Shortfall at 97.5%** rather than 99% VaR.

### Regulatory Context

- **Basel II/2.5:** 99% 10-day VaR, minimum 3× multiplier
- **Basel III / FRTB:** ES (97.5%) replaces VaR as the primary risk measure
- Kupiec and Christoffersen tests are the standard backtesting frameworks for regulatory reporting